# Метрики решений 1 и 4 с агрегацией по срезам датасетов

Гоняем метрики **решения 1** (делинеация neurokit -> диагноз по интервалам) и
**решения 4** (бустинг на ручных + neurokit фичах) и агрегируем их по разным
срезам всех датасетов: по **датасету** (подпапке), по **метке качества neurokit**
(zhao2018: Excellent / Barely acceptable / Unacceptable), по **полу** и **возрасту**.

Ноутбук самодостаточен: если в `data/` нет записей, создаёт синтетические
(с разным уровнем шума -> разное качество, в двух подпапках-`датасетах`).

## 0. Настройка

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
DATA_ROOT = os.path.abspath('../data')
LIMIT = None   # ограничить число записей для скорости (None = все)
print('data root:', DATA_ROOT)

## 1. Данные (или синтетика с разным качеством и в двух датасетах)

In [ ]:
from ecg_data import build_record_table, find_records

if not find_records(DATA_ROOT):
    print('Реальных записей нет — генерирую синтетические...')
    import scipy.io as sio, neurokit2 as nk, random
    random.seed(0)
    leads = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
    dx_pool = ['426783006','427084000','426627000','164889003','59118001']
    for k in range(24):
        ds = 'dataset_A' if k % 2 == 0 else 'dataset_B'     # два «датасета»
        noise = random.choice([0.02, 0.05, 0.3])            # разное качество
        outdir = os.path.join(DATA_ROOT, ds); os.makedirs(outdir, exist_ok=True)
        hr = random.choice([55, 75, 95, 115])
        sig = np.stack([nk.ecg_simulate(duration=10, sampling_rate=500, heart_rate=hr, noise=noise, random_state=i)[:5000] for i in range(12)])
        rid = f'{ds[-1]}{k:03d}'
        sio.savemat(os.path.join(outdir, f'{rid}.mat'), {'val': np.round(sig*1000).astype(np.int16)})
        hea = [f'{rid} 12 500 5000'] + [f'{rid}.mat 16 1000/mV 16 0 0 0 0 {l}' for l in leads] + [f'#Age: {random.randint(30,80)}', f'#Sex: {random.choice(["Male","Female"])}', f'#Dx: {random.choice(dx_pool)}']
        open(os.path.join(outdir, f'{rid}.hea'), 'w').write('\n'.join(hea)+'\n')
    print('готово')

df = build_record_table(DATA_ROOT, limit=LIMIT)
print('записей:', len(df), '| датасетов:', df['dataset'].nunique())
df[['record_id','dataset','age','sex','dx_codes']].head()

## 2. Решение 1 — делинеация + диагноз по интервалам

`run_pipeline` теперь возвращает и метку качества neurokit (`quality_label`) по
каждой записи — по ней потом делаем срез.

In [ ]:
from delineation import DelineationPipeline, run_pipeline, quality_metrics, delineation_failure_rate

pipe = DelineationPipeline(clean_method='neurokit', peaks_method='neurokit', delineate_method='dwt')
res1 = run_pipeline(df, pipeline=pipe, lead_index=1)   # лид II
print('надёжность делинеации (всего):', delineation_failure_rate(res1))
res1[['record_id','dataset','sex','age','quality_label','n_beats','predicted_labels','ground_truth_labels']].head()

In [ ]:
# общие метрики решения 1 (по каждой метке + micro)
m1 = quality_metrics(res1)
display(m1[m1['support'] > 0][['label','tp','fp','fn','precision','recall','f1','support']])
display(m1[m1['label'] == 'MICRO_AVERAGE'])

### 2.1 Агрегация метрик решения 1 по срезам

In [ ]:
def summarize_delineation_by(results, col):
    """micro precision/recall/F1 + доля падений делинеации по каждому значению среза."""
    rows = []
    for val, sub in results.groupby(col, dropna=False):
        micro = quality_metrics(sub)
        micro = micro[micro['label'] == 'MICRO_AVERAGE'].iloc[0]
        fail = delineation_failure_rate(sub)
        rows.append({col: val, 'n_records': len(sub), 'precision': micro['precision'],
                     'recall': micro['recall'], 'f1': micro['f1'],
                     'delin_fail_rate': fail['failure_rate']})
    return pd.DataFrame(rows)

# возрастные корзины для среза
res1['age_bin'] = pd.cut(pd.to_numeric(res1['age'], errors='coerce'),
                         [0, 40, 55, 70, 120], labels=['<40','40-55','55-70','70+'])

print('=== по датасету ===');       display(summarize_delineation_by(res1, 'dataset'))
print('=== по качеству neurokit ==='); display(summarize_delineation_by(res1, 'quality_label'))
print('=== по полу ===');           display(summarize_delineation_by(res1, 'sex'))
print('=== по возрасту ===');       display(summarize_delineation_by(res1, 'age_bin'))

## 3. Решение 4 — бустинг (ручные + neurokit фичи)

Извлекаем ручные фичи (или берём готовый `features.csv`), добавляем интервальные
фичи neurokit, обучаем бустинг и агрегируем метрики по тем же срезам. Качество
neurokit по записи берём из результатов делинеации (раздел 2) — без повторного счёта.

In [ ]:
from extract_features import extract_dataset, load_features
from train_boosting import build_feature_frame, build_xy, train_boosting, predict_proba, per_label_metrics
from ecg_data import CLASSES, make_split

FEATURES_CSV = os.path.abspath('../features.csv')
if not os.path.exists(FEATURES_CSV):
    print('Считаю ручные фичи (один раз)...')
    extract_dataset(df).to_csv(FEATURES_CSV, index=False)
hand_feats, hand_names = load_features(FEATURES_CSV)
feat_df, names = build_feature_frame(hand_feats, hand_names, df, os.path.abspath('../nk_features.csv'))
print('фич всего:', len(names), '(ручных', len(hand_names), '+ neurokit', len(names)-len(hand_names), ')')

In [ ]:
# сплит, обучение бустинга, предсказание по ВСЕМ записям (для срезов)
train_df, val_df, test_df = make_split(df)
if len(test_df) == 0:
    train_df, test_df = df.iloc[:int(len(df)*0.7)], df.iloc[int(len(df)*0.7):]
Xtr, Ytr, _ = build_xy(train_df, feat_df, names)
models, base = train_boosting(Xtr, Ytr, reweight=True)

Xall, Yall, ids = build_xy(df, feat_df, names)
probs = predict_proba(models, base, Xall)
print('предсказаны вероятности:', probs.shape)

In [ ]:
# общие метрики решения 4 по каждой метке
tbl4 = per_label_metrics(probs, Yall)
display(tbl4[tbl4['support'] > 0][['abbr','name','support','AUROC','AUPRC','precision','recall','f1']])
display(tbl4[tbl4['abbr'] == 'MACRO'])

### 3.1 Агрегация метрик решения 4 по срезам

In [ ]:
# срезы, выровненные по ids предсказаний
meta = df.set_index('record_id').loc[ids]
q_by_id = res1.set_index('record_id')['quality_label']
slices = pd.DataFrame({
    'dataset': meta['dataset'].values,
    'sex': meta['sex'].values,
    'quality_label': q_by_id.reindex(ids).values,
    'age_bin': pd.cut(pd.to_numeric(meta['age'], errors='coerce').values,
                      [0,40,55,70,120], labels=['<40','40-55','55-70','70+']),
})

def summarize_boosting_by(probs, Y, slice_series, threshold=0.5):
    s = pd.Series(slice_series).astype('object').fillna('NA').reset_index(drop=True)
    rows = []
    for val in pd.unique(s):
        mask = (s == val).to_numpy()
        if mask.sum() == 0: continue
        macro = per_label_metrics(probs[mask], Y[mask], threshold)
        macro = macro[macro['abbr'] == 'MACRO'].iloc[0]
        rows.append({'slice': val, 'n': int(mask.sum()),
                     'macro_AUROC': macro['AUROC'], 'macro_AUPRC': macro['AUPRC'], 'macro_F1': macro['f1']})
    return pd.DataFrame(rows).sort_values('n', ascending=False).reset_index(drop=True)

for col in ['dataset', 'quality_label', 'sex', 'age_bin']:
    print(f'=== решение 4, срез: {col} ===')
    display(summarize_boosting_by(probs, Yall, slices[col]))

## 4. Заметки

- Срез по **качеству neurokit** (`quality_label`) считается один раз в разделе 2
  (внутри делинеации) и переиспользуется для решения 4.
- Метрика решения 1 — micro precision/recall/F1 диагноза по интервалам vs реальные
  метки (+ доля падений самой делинеации по срезу). Решение 4 — macro AUROC/AUPRC/F1.
- На реальных данных срез по датасету покажет, где делинеация/бустинг деградируют,
  а срез по качеству — насколько метрики падают на «шумных» (Unacceptable) записях.